In [0]:
%run ../../common/_utils_config

In [0]:
%run ../../common/_utils_reference_data

In [0]:
%run ../utils/_utils_bronze_framework

In [0]:
%run ../utils/_utils_onboarding_tools

In [0]:
import copy

failures = []


def check(description, condition):
    print(("✅ " if condition else "❌ ") + description)
    if not condition:
        failures.append(description)


archive_url = load_source_spec_file("open_meteo_archive")["endpoints"][0]["url"]
forecast_url = load_source_spec_file("open_meteo_forecast")["endpoints"][0]["url"]
check("probes to the archive host use the open_meteo group", probe_governance(archive_url, {})[0] == "open_meteo")
check("probes to the forecast host use the open_meteo group", probe_governance(forecast_url, {})[0] == "open_meteo")
check("an unknown host gets its own group",
      probe_governance("https://example.org/data", {})[0] == "host_example_org")

_, _, _, probe_cost = probe_governance(archive_url, {"latitude": "40.4,37.3", "longitude": "-3.7,-5.9",
                                                     "hourly": "temperature_2m"})
check(f"a probe with 2 locations costs 2 (got {probe_cost})", probe_cost == 2)

# A probe to a blocked group makes no HTTP call
example_group = host_quota_group("example.org")
unblock_quota_group(example_group)
block_quota_group(example_group, utc_now() + timedelta(hours=1), "test block")
probe = probe_url("https://example.org/data")
print(probe)
check("probe to a blocked group is rejected", probe["status"] == "rejected" and "blocked" in probe["errors"][0])
check("no call recorded for the blocked group",
      spark.table(api_usage_table).filter(F.col("quota_group") == example_group).count() == 0)
unblock_quota_group(example_group)

In [0]:
base = copy.deepcopy(load_source_spec_file("ree"))
base["source_name"] = "ree_onboarding_test"

no_limits = copy.deepcopy(base)
no_limits.pop("limits", None)
outcome = json.loads(propose_source_spec(spec_to_yaml(no_limits)))
check("a new source without limits is rejected",
      outcome["status"] == "rejected" and any("limits is required" in e for e in outcome["errors"]))

conflicting = copy.deepcopy(base)
conflicting["limits"] = {"quota_group": "open_meteo", "per_minute": 10, "per_hour": 100, "per_day": 1000}
outcome = json.loads(propose_source_spec(spec_to_yaml(conflicting)))
check("limits that contradict the same quota group are rejected",
      outcome["status"] == "rejected" and any("must declare the same limits" in e for e in outcome["errors"]))

blocked_group = "test_onboarding_group"
blocked = copy.deepcopy(base)
blocked["limits"] = {"quota_group": blocked_group, "per_minute": 30, "per_hour": 500, "per_day": 3000}
unblock_quota_group(blocked_group)
block_quota_group(blocked_group, utc_now() + timedelta(hours=1), "test block")
outcome = json.loads(propose_source_spec(spec_to_yaml(blocked)))
check("a blocked provider stops the onboarding with status blocked (no approval asked)",
      outcome["status"] == "blocked")
unblock_quota_group(blocked_group)
check("the test source was not saved", "ree_onboarding_test" not in list_source_specs())

In [0]:
estimate = estimate_initial_load(load_source_spec_file("open_meteo_archive"))
print(estimate)
check("the initial load of open_meteo_archive costs more than one run's budget",
      estimate["calls"] > estimate["budget_per_run"])
check("so it is spread over at least 2 days", estimate["days"] >= 2)

In [0]:
if failures:
    raise AssertionError(f"{len(failures)} check(s) failed: {failures}")
print("✅ All onboarding governance checks passed")